# Phase 2: Target Encoding + Z-scores por Posición + Features Compuestas

Mejoras sobre Fase 1:
1. **Target encoding** para `School`, `Position`, `Position_Type` (con smoothing, dentro del fold para evitar data leakage)
2. **Z-scores por posición**: rendimiento relativo al grupo de la misma posición
3. **Features compuestas**: explosive score, agility composite, power score

Métrica objetivo: AUC

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

SEED    = 42
N_FOLDS = 5
TE_ALPHA = 10  # smoothing para target encoding (prior weight)

INPUT_PATH  = Path('/Users/haroldlagares/Downloads/competition/input')
RESULTS_DIR = Path('/Users/haroldlagares/Downloads/competition/results')
RESULTS_DIR.mkdir(exist_ok=True)

print('LightGBM version:', lgb.__version__)

## 2. Load Data

In [ ]:
train_raw = pd.read_csv(INPUT_PATH / 'train.csv')
test_raw  = pd.read_csv(INPUT_PATH / 'test.csv')

print('Train:', train_raw.shape)
print('Test: ', test_raw.shape)

## 3. Pipeline de Preprocessing

Encapsulamos todo en una función para poder llamarla tanto fuera del fold (test) como dentro (train/val).

In [ ]:
NULL_COLS = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
             'Broad_Jump', 'Agility_3cone', 'Shuttle']

PERF_COLS = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
             'Broad_Jump', 'Agility_3cone', 'Shuttle']

# Columnas donde menor valor = mejor rendimiento (tiempos)
INVERSE_COLS = ['Sprint_40yd', 'Agility_3cone', 'Shuttle']


def add_missing_flags(df):
    """Agrega columnas binarias para cada feature con NaN."""
    for col in NULL_COLS:
        df[f'missing_{col}'] = df[col].isna().astype(np.int8)
    return df


def group_impute(df, group_col, cols, medians):
    """Imputa cols con mediana por grupo. Fallback a mediana global."""
    df = df.copy()
    global_fallback = medians.median()
    for col in cols:
        mask = df[col].isna()
        if mask.any():
            group_vals = df.loc[mask, group_col].map(medians[col])
            # Donde el grupo no está en medians, usar fallback global
            group_vals = group_vals.fillna(global_fallback[col])
            df.loc[mask, col] = group_vals
    return df


def add_bmi(df):
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    return df


def add_zscores(df, pos_stats):
    """Z-score de cada PERF_COL dentro de la misma Position.
    pos_stats: dict {col: {'mean': Series, 'std': Series}} indexado por Position.
    """
    for col in PERF_COLS:
        mean_map = df['Position'].map(pos_stats[col]['mean'])
        std_map  = df['Position'].map(pos_stats[col]['std']).replace(0, np.nan).fillna(1)
        z = (df[col] - mean_map) / std_map
        # Invertir tiempos: menor tiempo = z más positivo = mejor
        if col in INVERSE_COLS:
            z = -z
        df[f'z_{col}'] = z
    return df


def add_composite_features(df):
    """Features compuestas a partir de z-scores."""
    # Explosive: velocidad explosiva — sprint, salto vertical, broad jump
    df['explosive_score'] = (df['z_Sprint_40yd'] + df['z_Vertical_Jump'] + df['z_Broad_Jump']) / 3
    # Agility: agilidad combinada
    df['agility_composite'] = (df['z_Agility_3cone'] + df['z_Shuttle']) / 2
    # Power: fuerza + salto vertical
    df['power_score'] = (df['z_Bench_Press_Reps'] + df['z_Vertical_Jump']) / 2
    # Overall athleticism (promedio de todos los z)
    z_cols = [f'z_{c}' for c in PERF_COLS]
    df['overall_athleticism'] = df[z_cols].mean(axis=1)
    return df


def label_encode_categoricals(df_train, df_test, cat_cols):
    """LabelEncoder fit en union de train+test."""
    from sklearn.preprocessing import LabelEncoder
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([df_train[col], df_test[col]], ignore_index=True).astype(str)
        le.fit(combined)
        df_train[col] = le.transform(df_train[col].astype(str))
        df_test[col]  = le.transform(df_test[col].astype(str))
        encoders[col] = le
    return df_train, df_test, encoders


print('Funciones de preprocessing definidas.')

## 4. Target Encoding

Para evitar data leakage, el target encoding se calcula **dentro de cada fold** usando solo el train_fold.
Las predicciones de test se promedian a través de los 5 folds.

Smoothing: `te = (n * mean_group + alpha * mean_global) / (n + alpha)`, alpha=10.

In [ ]:
def target_encode_static(train_df, apply_df, col, target='Drafted', alpha=10):
    """TE calculado en todo train_df, aplicado a apply_df.
    El smoothing (alpha) actúa como prior hacia la media global,
    controlando el leakage en grupos pequeños.
    """
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['te'] = (stats['mean'] * stats['count'] + global_mean * alpha) / (stats['count'] + alpha)
    return apply_df[col].map(stats['te']).fillna(global_mean)


TE_COLS = ['School', 'Position', 'Position_Type']
print('Target encoding estático con alpha =', TE_ALPHA)
print('Columnas:', TE_COLS)


## 5. Preprocessing Base (fuera del fold)

Todo lo que NO toca el target: missing flags, imputación, BMI, z-scores, label encoding de `Player_Type`.

In [ ]:
train = train_raw.copy()
test  = test_raw.copy()

# 1. Missing flags
train = add_missing_flags(train)
test  = add_missing_flags(test)

# 2. Imputación por grupo (medianas calculadas solo en train)
group_medians = train.groupby('Position_Type')[NULL_COLS].median()
train = group_impute(train, 'Position_Type', NULL_COLS, group_medians)
test  = group_impute(test,  'Position_Type', NULL_COLS, group_medians)

print('Nulos tras imputación (train):', train[NULL_COLS].isnull().sum().sum())
print('Nulos tras imputación (test): ', test[NULL_COLS].isnull().sum().sum())

# 3. BMI
train = add_bmi(train)
test  = add_bmi(test)

# 4. Z-scores por Position (stats calculadas solo en train)
pos_stats = {}
for col in PERF_COLS:
    pos_stats[col] = {
        'mean': train.groupby('Position')[col].mean(),
        'std':  train.groupby('Position')[col].std().replace(0, np.nan).fillna(1),
    }

train = add_zscores(train, pos_stats)
test  = add_zscores(test,  pos_stats)

# 5. Features compuestas
train = add_composite_features(train)
test  = add_composite_features(test)

# 6. Label encode solo Player_Type (3 valores, sin relación ordinal con target)
train, test, _ = label_encode_categoricals(train, test, ['Player_Type'])

print('Features tras preprocessing base:')
print([c for c in train.columns if c not in ['Id', 'Drafted', 'School', 'Position', 'Position_Type']])
# Target encoding estático (calculado en todo train, aplicado a train y test)
# Alpha=10 controla el shrinkage hacia la media global para grupos pequeños
for col in TE_COLS:
    te_col = f'te_{col}'
    train[te_col] = target_encode_static(train_raw, train, col, 'Drafted', TE_ALPHA)
    test[te_col]  = target_encode_static(train_raw, test,  col, 'Drafted', TE_ALPHA)

print('\nTE features agregadas. Ejemplos (te_School):')
print(train[['School', 'te_School']].drop_duplicates('School').sort_values('te_School').head(5))
print('...')
print(train[['School', 'te_School']].drop_duplicates('School').sort_values('te_School').tail(5))


## 6. LightGBM con Target Encoding dentro del Fold

In [ ]:
lgb_params = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         -1,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'verbose':           -1,
    'seed':              SEED,
}

FEATURE_COLS = [
    'Year', 'Age', 'Height', 'Weight',
    'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle',
    'Player_Type',
    'missing_Age', 'missing_Sprint_40yd', 'missing_Vertical_Jump', 'missing_Bench_Press_Reps',
    'missing_Broad_Jump', 'missing_Agility_3cone', 'missing_Shuttle',
    'BMI',
    'z_Sprint_40yd', 'z_Vertical_Jump', 'z_Bench_Press_Reps', 'z_Broad_Jump', 'z_Agility_3cone', 'z_Shuttle',
    'overall_athleticism',
    'te_School', 'te_Position', 'te_Position_Type',
]

TARGET_COL = 'Drafted'
y = train[TARGET_COL].values

X_train = train[FEATURE_COLS].values
X_test  = test[FEATURE_COLS].values

print(f'Features totales: {len(FEATURE_COLS)}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_scores = []
feature_importances = pd.DataFrame({'feature': FEATURE_COLS})

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURE_COLS)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=FEATURE_COLS, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    val_pred = model.predict(X_val)
    fold_auc = roc_auc_score(y_val, val_pred)
    fold_scores.append(fold_auc)

    oof_preds[val_idx] = val_pred
    test_preds += model.predict(X_test) / N_FOLDS

    feature_importances[f'fold_{fold}'] = model.feature_importance(importance_type='gain')

    print(f'Fold {fold} | AUC: {fold_auc:.5f} | Best iter: {model.best_iteration}')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOOF AUC: {oof_auc:.5f}  (mean folds: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f})')


## 7. Feature Importance

In [ ]:
fold_cols = [c for c in feature_importances.columns if c.startswith('fold_')]
feature_importances['mean_gain'] = feature_importances[fold_cols].mean(axis=1)
feature_importances = feature_importances.sort_values('mean_gain', ascending=False)

plt.figure(figsize=(11, 7))
plt.barh(feature_importances['feature'][:25], feature_importances['mean_gain'][:25])
plt.xlabel('Mean Gain')
plt.title('Top 25 Feature Importances — Phase 2')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(feature_importances[['feature', 'mean_gain']].to_string(index=False))

## 8. Guardar Submission

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
submission_path = RESULTS_DIR / f'submission_phase2_{timestamp}.csv'

submission = pd.read_csv(INPUT_PATH / 'sample_submission.csv')
submission['Drafted'] = test_preds
submission.to_csv(submission_path, index=False)

print(f'Submission guardado en: {submission_path}')
print(f'OOF AUC: {oof_auc:.5f}')
print(submission.head())